In [2]:
# Cell 1: Environment Setup, vLLM Launch, and Health Poll

# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv

# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip

# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("Virtual environment ready with vLLM installed!")

# 5. Launch vLLM server in background using venv binary
!/content/venv/bin/python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype half \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.85 \
    --port 8000 > vllm_server.log 2>&1 &

print("vLLM server process launched in background!")

# 6. Poll health endpoint until ready
import urllib.request
import time

health_url = "http://localhost:8000/v1/models"
start_time = time.time()
timeout = 300  # 5 minutes timeout

print("Polling server health endpoint...")
server_ready = False

while time.time() - start_time < timeout:
    try:
        req = urllib.request.Request(health_url)
        with urllib.request.urlopen(req) as response:
            if response.status == 200:
                print("\nServer is HEALTHY and ready to serve requests!")
                server_ready = True
                break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

if not server_ready:
    print("\nHealth poll timed out. Check logs using '!tail -n 30 vllm_server.log'")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Building depe

In [3]:
# Cell 2: The Seam Test
from openai import OpenAI

# Initialize client pointing to local vLLM endpoint
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

# Submit prompt targeting Qwen2.5-1.5B-Instruct
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {"role": "user", "content": "In one sentence, what is a GPU?"}
    ],
    max_tokens=64,
    temperature=0.0
)

print("=== Seam Test Output ===")
print(response.choices[0].message.content)

=== Seam Test Output ===
A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate the performance of graphics and computational tasks in computers and other devices.


In [4]:
# Cell 3: Restore Monday's Baseline
import json
import os
from google.colab import files

# Prompt upload if baselines.json isn't present
if not os.path.exists("baselines.json"):
    print("Please upload your baselines.json file from Day 2:")
    uploaded = files.upload()

with open("baselines.json", "r") as f:
    baseline = json.load(f)

print("Restored Baseline static batch throughput (tokens/s):")
print(json.dumps(baseline["batch"], indent=2))

Restored Baseline static batch throughput (tokens/s):
{
  "1": 35.2,
  "4": 53.3,
  "8": 104.8
}


In [5]:
# Cell 4: Async A/B Benchmark Sweep
import asyncio
import httpx
import time

FIXED_PROMPTS = [
    "Explain quantum computing in three short sentences.",
    "List five benefits of drinking enough water daily.",
    "Write a quick python function to reverse a string.",
    "What is the capital of France?",
    "Summarize the process of photosynthesis.",
    "Explain the difference between a process and a thread.",
    "What causes seasons to change on Earth?",
    "Give a short recipe for making chocolate chip cookies.",
]

async def send_request(client: httpx.AsyncClient, base_url: str, model: str, prompt: str) -> int:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 128,
        "temperature": 0.0,
    }
    res = await client.post(f"{base_url}/chat/completions", json=payload, timeout=60.0)
    data = res.json()
    return data["usage"]["completion_tokens"]

async def run_sweep(base_url: str, model: str, prompts: list, concurrencies: list):
    async with httpx.AsyncClient() as client:
        # Warmup request
        await send_request(client, base_url, model, "Warmup request to initialize GPU memory.")

        results = []
        for c in concurrencies:
            selected_prompts = (prompts * ((c // len(prompts)) + 1))[:c]

            start = time.perf_counter()
            tasks = [send_request(client, base_url, model, p) for p in selected_prompts]
            tokens_list = await asyncio.gather(*tasks)
            elapsed = time.perf_counter() - start

            total_tokens = sum(tokens_list)
            tok_per_s = round(total_tokens / elapsed, 2)

            results.append({"concurrency": c, "tokens_per_s": tok_per_s})
            print(f"Concurrency {c}: {tok_per_s} tokens/s")

        return results

# Execute sweep inside Colab asyncio loop
vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=FIXED_PROMPTS,
    concurrencies=[1, 4, 8],
)

Concurrency 1: 57.57 tokens/s
Concurrency 4: 130.67 tokens/s
Concurrency 8: 347.52 tokens/s


In [6]:
# Cell 5: Generate ab_report.json
import json

vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

speedup = {
    c: round(vllm_by_c[c] / base_by_c[c], 2)
    for c in vllm_by_c
    if c in base_by_c
}

# Enter the value recorded on your Prediction Card
PREDICTED_SPEEDUP = 1.5

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": PREDICTED_SPEEDUP,
}

with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Generated ab_report.json:")
print(json.dumps(report, indent=2))

Generated ab_report.json:
{
  "baseline": {
    "1": 35.2,
    "4": 53.3,
    "8": 104.8
  },
  "vllm": {
    "1": 57.57,
    "4": 130.67,
    "8": 347.52
  },
  "speedup_by_concurrency": {
    "1": 1.64,
    "4": 2.45,
    "8": 3.32
  },
  "predicted_speedup": 1.5
}


In [7]:
# Cell 6: Reflection & Scaling Ratio Analysis
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling = vllm_by_c[8] / vllm_by_c[1]

print("=== Engine Efficiency Analysis ===")
print(f"Static Batch Scaling (1 -> 8): {static_scaling:.2f}x")
print(f"vLLM Concurrency Scaling (1 -> 8): {vllm_scaling:.2f}x")
print(f"Continuous Batching Advantage: {vllm_scaling / static_scaling:.2f}x efficiency gain!")

=== Engine Efficiency Analysis ===
Static Batch Scaling (1 -> 8): 2.98x
vLLM Concurrency Scaling (1 -> 8): 6.04x
Continuous Batching Advantage: 2.03x efficiency gain!


In [9]:
# Cell 8: Green-Check Verifier (verify_cell.py)
import json
import os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str):
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    # keys may be strings or ints depending on how the report was built; normalise
    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    # the headline claim of the day
    if not vllm8 > base8:
        fail(
            f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
            f"batch-8 ({base8}); the engine swap should win here"
        )

    # speedup fields must be computed (present and numeric for at least c=8)
    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    # sanity: the reported speedup should match vllm8/base8 within rounding
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(
            f"speedup at 8 ({s8}) does not match vllm/baseline "
            f"({expected:.2f}); recompute it"
        )

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)

baseline batch-8: 104.8, vllm concurrency-8: 347.52
speedup at 8: 3.32x
GREEN CHECK: PASS
